In [ ]:
# Install required libraries (if not installed)
!pip install seaborn scikit-learn transformers --quiet

# Import required libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import classification_report

# Load dataset
file_path = "/content/SemEval2016-Task6-raw-annotations-stance.csv"  # Replace with your dataset path
df = pd.read_csv(file_path)

# Keep relevant columns
df = df[['Tweet', 'Stance']]

# Encode labels
label_encoder = LabelEncoder()
df['Stance'] = label_encoder.fit_transform(df['Stance'])

# Convert text to TF-IDF features with optimized settings
vectorizer = TfidfVectorizer(
    max_features=10000,  # Increase feature size
    stop_words='english',
    ngram_range=(1, 2),  # Use bigrams for better context
    sublinear_tf=True  # Apply sublinear term frequency scaling
)
X = vectorizer.fit_transform(df['Tweet'])
y = df['Stance']

# Split dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape}, Testing set: {X_test.shape}")
# Hyperparameter tuned SVM model
svm_model = SVC(kernel="rbf", C=2.0, gamma='scale', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

# Predict on test set
y_pred_svm = svm_model.predict(X_test)

# Evaluate SVM Model
svm_results = {
    "Accuracy": accuracy_score(y_test, y_pred_svm),
    "Precision": precision_score(y_test, y_pred_svm, average="weighted"),
    "Recall": recall_score(y_test, y_pred_svm, average="weighted"),
    "F1 Score": f1_score(y_test, y_pred_svm, average="weighted")
}

# Print Overall Results
print("\n🔹 Optimized SVM Model Performance (Overall):")
for metric, value in svm_results.items():
    print(f"{metric}: {value:.4f}")

# Per-Class Metrics
print("\n🔸 Classification Report (Per Class):")
print(classification_report(y_test, y_pred_svm, target_names=label_encoder.classes_, digits=4))

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Optimized SVM")
plt.show()
